In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

565

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
index = build_index(documents)

doc_idx = {doc["id"]: doc for doc in documents}

In [3]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [4]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"},
    )

In [5]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.pricing import PricingConfig
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

pricing_config = PricingConfig()
pricing_config.register_model("gpt-5.4-mini", 0.75, 4.50)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini", client=openai_client),
    pricing_config=pricing_config,
)

In [6]:
rec = ground_truth[0]
result = runner.loop(prompt=rec["question"])
result.last_message

'Yes — you can still join now. The course materials are available, and you can start anytime.\n\nOne important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [7]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({"name": message.name, "arguments": message.arguments})

    return tool_calls

In [8]:
tool_calls = extract_tool_calls(result.all_messages)
tool_calls

[{'name': 'search',
  'arguments': '{"query":"late to join course can I still join now too late enrollment late start FAQ"}'}]

In [9]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}
agent_result

{'question': 'I just found this course late — can I still join now, or is it too late?',
 'answer_agent': 'Yes — you can still join now. The course materials are available, and you can start anytime.\n\nOne important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late to join course can I still join now too late enrollment late start FAQ"}'}],
 'cost': Decimal('0.00109125'),
 'document': '74eb249bbf'}

In [10]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])
    tool_calls = extract_tool_calls(result.all_messages)

    return {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

In [11]:
from concurrent.futures import ThreadPoolExecutor

from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=3) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [12]:
df_agent = pd.DataFrame(agent_answers)
df_agent["cost"].sum()

Decimal('0.06341175')

In [13]:
df_agent.to_csv("data/agent-answers.csv", index=False)